In [ ]:
import pandas as pd

articles = pd.read_csv("/content/final.csv")
q_sent = pd.read_csv("/content/HDFC.csv")

# Drop rows with invalid dates in articles
article_date_col = next((col for col in articles.columns if 'date' in col.lower()), None)
quarter_date_col = 'Date'

articles[article_date_col] = pd.to_datetime(articles[article_date_col], errors='coerce')
q_sent[quarter_date_col] = pd.to_datetime(q_sent[quarter_date_col], errors='coerce')

articles = articles.dropna(subset=[article_date_col])

# Aggregate daily news sentiment (mean)
daily_news = (
    articles
    .groupby(articles[article_date_col].dt.date)['sentiment_score']
    .mean()
    .reset_index()
    .rename(columns={article_date_col: 'Date', 'sentiment_score': 'daily_news_sentiment'})
)
daily_news['Date'] = pd.to_datetime(daily_news['Date'])

# Keep all columns from HDFC.csv, merge on Date
final_df = pd.merge(q_sent, daily_news, on='Date', how='left')

final_df['daily_news_sentiment'] = final_df['daily_news_sentiment'].fillna(0)

output_path = 'final_sentiment.csv'
final_df.to_csv(output_path, index=False)

print(final_df.head())


In [ ]:
final_df.dropna()

In [ ]:
!pip install stable-baselines3[extra] --quiet

import gym
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gymnasium import spaces
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.env_checker import check_env


In [ ]:
class StockTradingEnv(gym.Env):
    def __init__(self, df, window_size=5, initial_balance=10000):
        super(StockTradingEnv, self).__init__()
        self.df = df.reset_index(drop=True)
        self.window_size = window_size
        self.initial_balance = initial_balance

        self.df["Close"] = pd.to_numeric(self.df["Close"], errors="coerce")
        self.df = self.df.dropna(subset=["Close"])

        self.feature_columns = [col for col in df.columns if col not in ['Date', 'quarter'] and df[col].dtype != 'O']
        self.num_features = len(self.feature_columns)

        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf,
                                            shape=(1 + window_size * self.num_features,), dtype=np.float32)
        self.reset()

    def _get_observation(self):
        window = self.df.loc[self.current_step - self.window_size:self.current_step - 1, self.feature_columns].values.flatten()
        obs = np.concatenate(([self.position], window), axis=0)
        return obs.astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = self.window_size
        self.balance = self.initial_balance
        self.position = 0
        self.entry_price = 0
        self.total_profit = 0
        self.trades = 0
        self.trade_history = []
        return self._get_observation(), {}

    def step(self, action):
        terminated = False
        truncated = False
        reward = 0

        price = self.df.loc[self.current_step, "Close"]

        if action == 1 and self.position == 0:  # Buy
            self.position = 1
            self.entry_price = price
            self.trade_history.append(('Buy', self.current_step, price))

        elif action == 2 and self.position == 1:  # Sell
            profit = price - self.entry_price
            reward = profit
            self.total_profit += profit
            self.position = 0
            self.entry_price = 0
            self.trades += 1
            self.trade_history.append(('Sell', self.current_step, price))

        self.current_step += 1
        if self.current_step >= len(self.df) - 1:
            terminated = True

        return self._get_observation(), reward, terminated, truncated, {}


In [ ]:
'''
class TrainLoggerCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self.losses = []
        self.current_reward = 0

    def _on_step(self) -> bool:
        if self.locals["dones"][0]:
            self.episode_rewards.append(self.current_reward)
            self.current_reward = 0
        self.current_reward += self.locals["rewards"][0]
        if "loss" in self.locals:
            self.losses.append(self.locals["loss"].item())
        return True
'''

class TrainLoggerCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self.losses = []
        self.current_reward = 0

    def _on_step(self) -> bool:
        if self.locals["dones"][0]:
            self.episode_rewards.append(self.current_reward)
            self.current_reward = 0
        self.current_reward += self.locals["rewards"][0]

        # Track loss from the replay buffer training step
        if "infos" in self.locals:
            info = self.locals["infos"][0]
            if "loss" in info:
                self.losses.append(info["loss"])

        return True



In [ ]:
# Convert sentiment columns to numeric
final_df["report_sentiment"] = pd.to_numeric(final_df["report_sentiment"], errors="coerce")
final_df["daily_news_sentiment"] = pd.to_numeric(final_df["daily_news_sentiment"], errors="coerce")


In [ ]:
#Apply a rolling z‑score to both report_sentiment and daily_news_sentiment so they’re comparable


for col in ["report_sentiment", "daily_news_sentiment"]:
    rolling_mean = final_df[col].rolling(window=20, min_periods=1).mean()
    rolling_std = final_df[col].rolling(window=20, min_periods=1).std(ddof=0)
    final_df[f"{col}_z"] = ((final_df[col] - rolling_mean) / rolling_std).clip(-3, 3)

In [ ]:
final_df

In [ ]:
cols_to_numeric = ["Open", "High", "Low", "Close", "Adj Close", "Volume", "report_sentiment", "daily_news_sentiment"]
for col in cols_to_numeric:
    final_df[col] = pd.to_numeric(final_df[col], errors="coerce")


In [ ]:
# cleaning - remove commas and convert to nuneric
final_df["Volume"] = final_df["Volume"].astype(str).str.replace(",", "").astype(float)


In [ ]:
# Momentum indicator over 5, 10, 20 
for n in [5, 10, 20]:
    final_df[f"momentum_{n}"] = final_df["Close"].pct_change(n)

for n in [10, 20]:
    final_df[f"volatility_{n}"] = final_df["Close"].pct_change().rolling(window=n).std()

final_df["price_change"] = final_df["Close"].pct_change()
final_df["price_change"] = pd.to_numeric(final_df["price_change"], errors="coerce")
final_df["price_change"] = final_df["price_change"].astype(str).str.replace(",", "").astype(float)
final_df["volume_flow"] = final_df["price_change"] * final_df["Volume"]
final_df["volume_flow_ma10"] = final_df["volume_flow"].rolling(window=10).mean()


In [ ]:

K = 5
for col in ["report_sentiment_z", "daily_news_sentiment_z"]:
    for lag in range(1, K + 1):
        final_df[f"{col}_lag{lag}"] = final_df[col].shift(lag)


In [ ]:

final_df_clean = final_df.dropna().reset_index(drop=True)


In [ ]:
#final_df_clean.to_csv("rl_ready_sentiment_features.csv", index=False)
final_df_clean

In [ ]:
final_df_clean=final_df_clean.drop(['Adj Close','Date','Volume','quarter'],axis="columns")
final_df_clean

In [ ]:
from sklearn.preprocessing import StandardScaler

# we only want numeric features
feature_columns = [col for col in final_df_clean.columns]

#leads to much better results as all different columns have differnt value ranges
scaler = StandardScaler()
final_df_clean[feature_columns] = scaler.fit_transform(final_df_clean[feature_columns])


In [ ]:
#final_df_clean
final_df_clean.to_csv("rl_ready_features.csv", index=False)

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback

class LossLoggerCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.policy_losses = []
        self.value_losses = []
        self.entropy_losses = []

    def _on_step(self) -> bool:
        for key, value in self.logger.name_to_value.items():
            if key == "train/policy_loss":
                self.policy_losses.append(value)
            elif key == "train/value_loss":
                self.value_losses.append(value)
            elif key == "train/entropy_loss":
                self.entropy_losses.append(value)
        return True

loss_logger = LossLoggerCallback()

In [ ]:
from stable_baselines3.common.callbacks import CallbackList

train_logger = TrainLoggerCallback()

loss_logger = LossLoggerCallback()

combined_callback = CallbackList([train_logger, loss_logger])

In [ ]:
# Upload your preprocessed CSV with correct columns
from stable_baselines3 import A2C

df2 = pd.read_csv("/content/rl_ready_features.csv")
df2 = df2.dropna()

env = StockTradingEnv(df2, window_size=5)
check_env(env)

model = A2C(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=1e-4,
    n_steps=5,               # How many steps before updating
    gamma=0.99,
    ent_coef=0.01            # Entropy regularization
)


steps_per_episode = len(df2) - 5 - 1
total_timesteps = steps_per_episode * 200
model.learn(total_timesteps=total_timesteps, callback=combined_callback)

model.save("a2c_hdfc_model_changed")


In [ ]:
import matplotlib.pyplot as plt

rewards = train_logger.episode_rewards
#losses = loss_logger.losses

def smooth(data, weight=0.9):
    if not data:
        return []
    smoothed = []
    last = data[0]
    for point in data:
        smoothed_val = last * weight + (1 - weight) * point
        smoothed.append(smoothed_val)
        last = smoothed_val
    return smoothed

smoothed_rewards = smooth(rewards)
smoothed_losses = smooth(losses)

plt.figure(figsize=(12, 4))
plt.plot(rewards, label="Raw Rewards", alpha=0.4)
plt.plot(smoothed_rewards, label="Smoothed Rewards", linewidth=2)
plt.title("Episode Rewards over Time")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.legend()
plt.grid(True)
plt.show()

import matplotlib.pyplot as plt

def smooth(data, weight=0.9):
    if not data:
        return []
    smoothed = [data[0]]
    for point in data[1:]:
        smoothed.append(smoothed[-1] * weight + (1 - weight) * point)
    return smoothed

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.plot(loss_logger.policy_losses, alpha=0.5, label="Loss")
plt.plot(smooth(loss_logger.policy_losses), label="Smoothed Loss", linewidth=2)
plt.title("Loss")
plt.grid(True)
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(loss_logger.value_losses, alpha=0.5, label="Value Loss")
plt.plot(smooth(loss_logger.value_losses), label="Smoothed", linewidth=2)
plt.title("Value Loss")
plt.grid(True)
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(loss_logger.entropy_losses, alpha=0.5, label="Entropy")
plt.plot(smooth(loss_logger.entropy_losses), label="Smoothed", linewidth=2)
plt.title("Entropy Bonus (Exploration)")
plt.grid(True)
plt.legend()

plt.suptitle("A2C Training Loss Components")
plt.tight_layout()
plt.show()

